In [1]:
import pandas as pd
import os
from prefect import task, get_run_logger

def address_to_fips(task_name, df, address_column="address", merge_columns=[]):

    # Add GOOGLE_MAPS_API_KEY to your .env file
  API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "")

  # documentation: https://geopy.readthedocs.io/en/latest/#geopy.geocoders.GoogleV3
  from geopy.geocoders import GoogleV3
  geolocator = GoogleV3(api_key=API_KEY)

  from geopy.extra.rate_limiter import RateLimiter
  geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.1)

  import addfips
  af = addfips.AddFIPS()
  
  logger = get_run_logger()
  logger.info(f"🔵 [{task_name}] Starting...")

  fips_df = pd.DataFrame(columns=["geoid", "state_name", "state_fips", "county_name", "county_fips"])
  cannot_geocode = []
  cannot_geoid = []

  for index, row in df.iterrows():
    try:
      info = geocode(row[address_column]).raw
      addy_components = info['address_components']
      county = next(name["long_name"] for name in addy_components if 'administrative_area_level_2' in name["types"])
      state = next(name["long_name"] for name in addy_components if 'administrative_area_level_1' in name["types"])
    except:
      cannot_geocode = cannot_geocode + [str(index) + "\t" + str(row[address_column])]
      result = {"geoid": None, "state_name": None, "state_fips": None, "county_name": None, "county_fips": None}
      fips_df.loc[index] = result
      continue
    try:
      geoid = af.get_county_fips(county, state)
      result = {"geoid": geoid, "state_name": state, "state_fips": geoid[:2], "county_name": county, "county_fips": geoid[2:]}
    except:
      cannot_geoid = cannot_geoid + [str(index) + "\t" + str(row[address_column]) + "\t" + str(county)]
      result = {"geoid": None, "state_name": None, "state_fips": None, "county_name": None, "county_fips": None}
    fips_df.loc[index] = result

  # print warnings
  if len(cannot_geocode) > 0:
    logger.info(f"Could not geocode these addresses:")
    for address in cannot_geocode:
        logger.info(address)
  if len(cannot_geoid) > 0:
    print(f"Could not find the geoid of these addresses:")
    for address in cannot_geoid:
        logger.info(address)
  return fips_df




In [4]:
df = pd.read_csv("./Biodiesel_Plants.csv")
# address_to_fips(df)

In [1]:
from typing import Optional
import pandas as pd
from prefect import task, get_run_logger
from ca_biositing.pipeline.utils.gdrive_to_pandas import gdrive_to_df
import os

# @task
def extract(project_root: Optional[str] = None) -> Optional[pd.DataFrame]:
    """
    Extracts raw data from a .csv file.

    This function serves as the 'Extract' step in an ETL pipeline. It connects
    to the data source and returns the data as is, without transformation.

    Returns:
        A pandas DataFrame containing the raw data, or None if an error occurs.
    """
    logger = get_run_logger()

    FILE_NAME = "Biodiesel_Plants.csv"
    MIME_TYPE = "text/csv"
    CREDENTIALS_PATH = os.getenv("CREDENTIALS_PATH", "credentials.json")
    DATASET_FOLDER = "src/ca_biositing/pipeline/ca_biositing/pipeline/temp_external_datasets/"
    print(f"Extracting raw data from '{FILE_NAME}'...")

    # If project_root is provided (e.g., from a notebook), construct an absolute path
    # Otherwise, use the default relative path (for the main pipeline)
    credentials_path = CREDENTIALS_PATH
    dataset_folder = DATASET_FOLDER
    if project_root:
        credentials_path = os.path.join(project_root, CREDENTIALS_PATH)
        dataset_folder = os.path.join(project_root, DATASET_FOLDER)

    # The gdrive_to_df function handles authentication, data fetching, and error handling.
    raw_df = gdrive_to_df(FILE_NAME, MIME_TYPE, credentials_path, dataset_folder)



    if raw_df is None:
        print("Failed to extract data. Aborting.")
        return None

    print("Successfully extracted raw data.")
    return raw_df

# def get_project_root():
#     return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
 
 
# project_root = get_project_root()
biodiesel_df = extract()
biodiesel_df


Extracting raw data from 'Biodiesel_Plants.csv'...
An unexpected error occurred: [Errno 2] No such file or directory: 'credentials.json'
Failed to extract data. Aborting.


In [14]:
from geopy.geocoders import GoogleV3
import os
from prefect import task, get_run_logger

import numpy as np
import pandas as pd

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "")
geolocator = GoogleV3(api_key=API_KEY)


# for other python libraries like geocoder you can do time.sleep i think
# max rate is 50 requests per second
from geopy.extra.rate_limiter import RateLimiter
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.1)



import addfips
af = addfips.AddFIPS()

 # warning: can be slow for VERY LONG dataframes, and returns "None" for misspelled names.
 # I've added warning handling for the latter problem

address_types = ['street_number', 'route', "intersection", "natural_feature", "airport", "park", "point_of_interest", 'post_box', 'landmark']




# scour the json for the appropriate address component
def get_component(name_length, comp_type, addy_components):
  return next((name[name_length] for name in addy_components if comp_type in name["types"]), None)




# main function; need to define names of columns where address and latlong are stored
def parse_addresses(df, address_column="address", merge_columns=[], lat="latitude", long="longitude"):

  try:
    logger = get_run_logger()
  except Exception:
    import logging
    logger = logging.getLogger(__name__)

  # if address info is distributed among several columns, merge them together
  if len(merge_columns) > 0:
    df['is_na'] = df[merge_columns].isnull().sum(axis=1) >= 3 # if there are three or more columns with NaN values it's invalid
    df[address_column] = df[merge_columns].apply(lambda row: ', '.join(row.fillna('').values.astype(str)), axis=1)

  else:

    df['is_na'] = df[address_column].isnull()

  address_df = pd.DataFrame(columns=["closest_address_line_1", "closest_address_line_2", "closest_city", "closest_county", "closest_state", "closest_postal_code", "closest_latitude", "closest_longitude"])
  geoid_df = pd.DataFrame(columns=["closest_geoid", "closest_state_name", "closest_state_fips", "closest_county_name", "closest_county_fips"])

  # put weird addresses in an array for displaying in a warning
  unparseable = []
  for index, row in df.iterrows():
    try:
      if row['is_na']:
        # categorize as unparseable
        raise Exception

      info = geocode(row[address_column]).raw
      addy_components = info['address_components']


      # idea for address:
      # - get component types
      # - iterate through component types and see if they match the following type
      # - add to address if it exists

      # get address components
      address = ""
      row_types = np.concatenate([name["types"] for name in addy_components])
      for possible_type in address_types:
        if possible_type in row_types and get_component("long_name", possible_type, addy_components) != None:
          address = address + get_component("long_name", possible_type, addy_components) + " "
          if possible_type != "street_number" and possible_type != "route":
            print(get_component("long_name", possible_type, addy_components), possible_type)

      county = get_component("long_name", "administrative_area_level_2", addy_components)
      city = get_component("short_name", "locality", addy_components)
      state = get_component("short_name", "administrative_area_level_1", addy_components)
      zip_code = get_component("short_name", "postal_code", addy_components)
      zip_suffix = get_component("short_name", "postal_code_suffix", addy_components)

      # get latitude and longitude
      if isinstance(row[lat], float) and isinstance(row[long], float):
        latitude = row[lat]
        longitude = row[long]
      else:
        latitude = info['geometry']['location']['lat'] if isinstance(info['geometry']['location']['lat'], float) else None
        longitude = info['geometry']['location']['lng'] if isinstance(info['geometry']['location']['lng'], float) else None
        

      address_result = {"closest_address_line_1": address, "closest_address_line_2": None, "closest_city": city, "closest_county": county, "closest_state": state, "closest_postal_code": zip_code + "-" + zip_suffix if (isinstance(zip, str) and isinstance(zip_suffix, str)) else None, "closest_latitude": latitude, "closest_longitude": longitude}

      # get geoids
      geoid = af.get_county_fips(county, state) if (isinstance(county, str) and isinstance(state, str)) else None
      state_fips = geoid[:2] if isinstance(geoid, str) else None
      county_fips = geoid[2:] if isinstance(geoid, str) else None
      geoid_result = {'closest_geoid': geoid, "closest_state_name": state, "closest_state_fips": state_fips, "closest_county_name": county, "closest_county_fips": county_fips}

      if geoid is None:
        unparseable = unparseable + [str(index) + "\t" + str(row[address_column])]

    except:
      # handle weird addresses
      unparseable = unparseable + [str(index) + "\t" + str(row[address_column])]
        
      if isinstance(row[lat], float) and isinstance(row[long], float):
        latitude = row[lat]
        longitude = row[long]
      else:
        latitude = info['geometry']['location']['lat'] if isinstance(info['geometry']['location']['lat'], float) else None
        longitude = info['geometry']['location']['lng'] if isinstance(info['geometry']['location']['lng'], float) else None

      address_result = {"closest_address_line_1": None, "closest_address_line_2": None, "closest_city": None, "closest_county": None, "closest_state": None, "closest_postal_code": None, "closest_latitude": latitude, "closest_longitude": longitude}
      geoid_result = {'closest_geoid': "00000", "closest_state_name": None, "closest_state_fips": None, "closest_county_name": None, "closest_county_fips": None}

    address_df.loc[index] = address_result
    geoid_df.loc[index] = geoid_result

  # convert to geodataframe
  # projection is NAD83 / California Albers, which is what USGS uses: https://spatialreference.org/ref/epsg/3310/
  # address_gdf = gpd.GeoDataFrame(
  #   address_df, geometry=gpd.points_from_xy(address_df["longitude"], address_df['latitude']), crs="EPSG:3310"
  # )

  # print weird addresses
  if len(unparseable) > 0:
    logger.warn(f"Some addresses were unparseable: \n{"\n".join(unparseable)}")


  # returns a gdf and a df that need to be added to the database
  return address_df, geoid_df

In [16]:
"""
ETL Transform Template.


This module provides a template for transforming raw data from multiple sources.
It includes standard cleaning, coercion, and normalization patterns used in the pipeline.
"""

import pandas as pd
import numpy as np
from typing import List, Optional, Dict
from prefect import task, get_run_logger
from ca_biositing.pipeline.utils.cleaning_functions import cleaning as cleaning_mod
from ca_biositing.pipeline.utils.cleaning_functions import coercion as coercion_mod
from ca_biositing.pipeline.utils.name_id_swap import normalize_dataframes
from ca_biositing.pipeline.utils.geo_utils import parse_addresses


# --- CONFIGURATION ---
# List the names of the extract modules this transform depends on.
# The pipeline runner provides these in the `data_sources` dictionary.
EXTRACT_SOURCES: List[str] = ["biodiesel_plants"]

@task
def transform(
    data_sources: Dict[str, pd.DataFrame],
    etl_run_id: int = None,
    lineage_group_id: int = None
) -> Optional[pd.DataFrame]:
    """
    Transforms raw data from multiple sources into a structured format.

    Args:
        data_sources: Dictionary where keys are source names and values are DataFrames.
        etl_run_id: ID of the current ETL run.
        lineage_group_id: ID of the lineage group.
    """
    try:
        logger = get_run_logger()
    except Exception:
        import logging
        logger = logging.getLogger(__name__)

    # CRITICAL: Lazy import models inside the task to avoid Docker import hangs
    from ca_biositing.datamodels.models import (
        LocationAddress,
        Place
        # Add other models needed for normalization here (e.g., Resource, Unit)
    )

    # 1. Input Validation
    for source_name in EXTRACT_SOURCES:
        if source_name not in data_sources:
            logger.error(f"Required data source '{source_name}' not found.")
            return None

    logger.info(f"Transforming data from sources: {EXTRACT_SOURCES}")

    # 2. Cleaning & Coercion
    processed_dfs = []
    for source_name in EXTRACT_SOURCES:
        df = data_sources[source_name].copy()

        if df.empty:
            continue

        # Standardize column names (snake_case) and basic string cleaning
        cleaned_df = cleaning_mod.standard_clean(df)

        # Add lineage tracking metadata
        cleaned_df['etl_run_id'] = etl_run_id
        cleaned_df['lineage_group_id'] = lineage_group_id

        # if address = null add state + city
        cleaned_df["address"] = np.where(cleaned_df["address"].isna(), cleaned_df["city"] + " " + cleaned_df["state"], cleaned_df["address"])

        # Coerce data types (Update these lists based on your schema)
        coerced_df = coercion_mod.coerce_columns(
            cleaned_df,
            int_cols=["capacity_mmg_per_y", "bbi_index"],
            float_cols=[],
            datetime_cols=['created_at', 'updated_at']
        )
        
        processed_dfs.append(coerced_df)

    if not processed_dfs:
        return pd.DataFrame()

    # Combine sources if necessary, or handle them individually
    combined_df = pd.concat(processed_dfs, ignore_index=True)

    address_df, geoid_df = parse_addresses(combined_df, address_column="address", lat="latitude", long="longitude")

    added_address_df = pd.concat([combined_df, address_df, geoid_df], axis=1)

    # 3. Normalization (Name-to-ID Swapping)
    # Format: 'dataframe_column': (SQLAlchemyModel, 'lookup_field_in_db')
    normalize_columns = {

    }

    

     # Manual normalization for Place (County) to avoid NotNullViolation on geoid
    # and provide a resilient lookup that defaults to state-level GEOID.
    from ca_biositing.pipeline.utils.geo_utils import get_geoid
    from sqlmodel import Session, select
    from ca_biositing.pipeline.utils.engine import engine

    with Session(engine) as session:
        places = session.exec(select(Place.geoid, Place.county_name)).all()
        county_to_geoid = {p.county_name.lower(): p.geoid for p in places if p.county_name}

    logger.info("Normalizing data (swapping names for IDs)...")
    normalized_df = normalize_dataframes(added_address_df, normalize_columns)[0]


    # Bridge County (Place) to LocationAddress
    # We need to find or create a generic LocationAddress for each County
    if 'closest_geoid' in normalized_df.columns:
        logger.info("Bridging County (Place) to LocationAddress...")
        from sqlmodel import Session, select
        from ca_biositing.pipeline.utils.engine import engine

        with Session(engine) as session:
            # Get unique county_ids (these are geoids from Place table)
            county_ids = normalized_df['closest_geoid'].dropna().unique()
            place_to_address_map = {}

            for index, row in normalized_df.iterrows():
                # Find or create LocationAddress and Place where geography_id = geoid
                geoid = row["closest_geoid"]
                if geoid != None:
                    stmt1 = select(Place).where(
                        Place.geoid == geoid
                    )
                    place = session.exec(stmt1).first()
                    
                    stmt2 = select(LocationAddress).where(
                        LocationAddress.geography_id == geoid,
                        # LocationAddress.address_line1 == None
                    )
                    address = session.exec(stmt2).first()

                    if not place:
                        logger.info(f"Creating new Place for county geoid: {geoid}")
                        place = Place(
                            geoid=geoid,
                            state_name=row["closest_state_name"],
                            state_fips=row["closest_state_fips"],
                            county_name=row["closest_county_name"],
                            county_fips=row["closest_county_fips"],
                        )
                        session.add(place)
                        session.flush()

                    if not address:
                        logger.info(f"Creating new generic LocationAddress for county geoid: {geoid}")
                        
                        address = LocationAddress(
                            geography_id=geoid,
                            address_line1=row["closest_address_line_1"],
                            address_line2=row["closest_address_line_2"],
                            city=row["closest_city"],
                            zip=row["closest_postal_code"],
                            lat=row["closest_latitude"],
                            lon=row["closest_longitude"],
                            is_anonymous=False
                            )
                        session.add(address)
                        session.flush()
    
                    place_to_address_map[geoid] = address.id

            session.commit()

            # Map county_id (Place.geoid) to sampling_location_id (LocationAddress.id)
            normalized_df['address'] = normalized_df['closest_geoid'].map(place_to_address_map)
            logger.info(f"Mapped {len(place_to_address_map)} counties to LocationAddresses")


    # 4. Column Renaming
    # TODO: Update this dictionary to match your source-to-target mapping
    
    rename_columns = {
        
    }
    normalized_df = normalized_df.rename(columns=rename_columns)

    # 5. Final Mapping & Selection
    # TODO: Update this list to match the columns in your target database table
    try:
        # Ensure lineage columns exist even if not provided in input
        if etl_run_id:
            normalized_df['etl_run_id'] = etl_run_id
        if lineage_group_id:
            normalized_df['lineage_group_id'] = lineage_group_id

        final_df = normalized_df[[
            "company",
            "bbi_index",
            "city",
            "state",
            "capacity_mmg_per_y",
            "feedstock",
            "status",
            "address", 
            "coordinates",
            "latitude",
            "longitude",
            "source",
            'etl_run_id',
            'lineage_group_id',
        ]].copy()

        logger.info(f"Successfully transformed {len(final_df)} records.")
        return final_df

    except KeyError as e:
        logger.error(f"Missing required column during transform: {e}")
        return normalized_df

biodiesel_df = pd.read_csv("./Biodiesel_Plants.csv")
transformed_df = transform({"biodiesel_plants": biodiesel_df})


16:18:50.215 | INFO    | Task run 'transform' - Transforming data from sources: ['biodiesel_plants']

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\src\ca_biositing\pipeline\ca_biositing\pipeline\utils\cleaning_functions\cleaning.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.astype("object").replace(regex, np.nan, regex=True)


Wisconsin 80 & Mauston Road intersection


C:\Users\Abigail\AppData\Local\Temp\ipykernel_36772\924996609.py:135: DeprecationWarning: The 'warn' method is deprecated, use 'warning' instead
  logger.warn(f"Some addresses were unparseable: \n{"\n".join(unparseable)}")


16:18:58.070 | WARNING | Task run 'transform' - Some addresses were unparseable: 
0       new haven connecticut

...

16:18:59.963 | INFO    | Task run 'transform' - Creating new generic LocationAddress for county geoid: 13115

16:18:59.993 | INFO    | Task run 'transform' - Mapped 67 counties to LocationAddresses

16:19:00.015 | INFO    | Task run 'transform' - Successfully transformed 78 records.

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


16:19:00.033 | INFO    | Task run 'transform' - Finished in state Completed()

In [17]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from prefect import task, get_run_logger
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy import select
from sqlalchemy.orm import Session
from ca_biositing.pipeline.utils.engine import get_engine
from ca_biositing.pipeline.utils.geo_utils import get_geoid

@task
def load(df: pd.DataFrame) -> bool:
    """
    Upserts BiodieselPlant records into the database.
    Returns True on success, False on failure.
    """
    logger = get_run_logger()

    if df is None or df.empty:
        logger.info("No data to load.")
        return True

    logger.info(f"Upserting {len(df)} records...")

    try:
        # Lazy import to avoid Docker import hangs
        from ca_biositing.datamodels.models import InfrastructureBiodieselPlants
        from ca_biositing.datamodels.models import LocationAddress, Place
    
        now = datetime.now(timezone.utc)
    
        # Filter columns to match the table schema
        table_columns = {c.name for c in InfrastructureBiodieselPlants.__table__.columns}
        records = df.replace({np.nan: None}).to_dict(orient="records")
    
        engine = get_engine()
        with engine.connect() as conn:
            with Session(bind=conn) as session:
                
                for i, record in enumerate(records):
                    if i > 0 and i % 500 == 0:
                        logger.info(f"Processed {i} records...")
    
                    # Keep only columns that exist in the model
                    clean_record = {k: v for k, v in record.items() if k in table_columns}
                     # Handle timestamps
                    clean_record['updated_at'] = now
                    if clean_record.get('created_at') is None:
                        clean_record['created_at'] = now
    
                    # Build Upsert Statement (PostgreSQL specific)
                    stmt = insert(InfrastructureBiodieselPlants).values(clean_record)
    
                    # Define columns to update on conflict
                    # Exclude primary keys and creation timestamps
                    update_dict = {
                        c.name: stmt.excluded[c.name]
                        for c in InfrastructureBiodieselPlants.__table__.columns
                        if c.name not in ['biodiesel_plant_id', 'created_at', 'record_id']
                    }
    
                    upsert_stmt = stmt.on_conflict_do_update(
                        index_elements=['biodiesel_plant_id'], # Replace with your unique constraint column
                        set_=update_dict
                    )
    
                    session.execute(upsert_stmt)
    
                session.commit()
        logger.info("Successfully upserted records.")
        return True
    except Exception as e:
        logger.error(f"Failed to load records: {e}")
        return False

load(transformed_df)

16:19:10.334 | INFO    | Task run 'load' - Upserting 78 records...

16:19:11.011 | INFO    | Task run 'load' - Successfully upserted records.

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


16:19:11.022 | INFO    | Task run 'load' - Finished in state Completed()

True